# Train the biomedical truth-verification model

This notebook trains the local **BiomedBERT truth verifier** with HealthFC, BioASQ 7B, and SciFact. It does not train a check-worthiness classifier, and it does not use the old medicine/cosmetic lookup logic.

Before starting, select **Runtime > Change runtime type > T4 GPU** in Colab. Keep the four source archives in the project's `data` folder. Do not edit the held-out test data after looking at its results.

In [ ]:
# Connect Colab to your Google Drive. Approve the prompt shown by Google.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# EDIT ONLY THIS PATH if your project folder has a different name or location.
import os
from pathlib import Path
PROJECT_DIR = Path('/content/drive/MyDrive/i-am-building-a-final-year')
assert PROJECT_DIR.is_dir(), f'Project folder not found: {PROJECT_DIR}'
os.chdir(PROJECT_DIR)
print('Using project:', PROJECT_DIR)

In [ ]:
# Confirm the real datasets are present before installing or training anything.
required = [
    'data/HealthFC.csv',
    'data/BioASQ-training7b.zip',
    'data/Task7BGoldenEnriched.zip',
    'data/data.tar.gz',
]
missing = [name for name in required if not (PROJECT_DIR / name).is_file()]
assert not missing, 'Missing dataset files: ' + ', '.join(missing)
print('All four dataset files were found.')

In [ ]:
# Install only the packages needed for data preparation, training, and tests.
# The pretrained BiomedBERT weights are downloaded from Hugging Face later.
%pip install -q -e '.[local-ml,dev]'

In [ ]:
# Run fast tests before spending GPU time. A failure here must be fixed first.
!python -m unittest discover -s tests -v
!python -m compileall -q src tests app.py

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: switch the Colab runtime to a GPU before training.')

## Prepare one auditable schema

This step excludes HealthFC explanations, keeps only BioASQ yes/no items, excludes the unlabeled SciFact test set, and records source hashes and original splits. The output counts are **not accuracy metrics**.

In [ ]:
!biofc-prepare-datasets --data-dir data --output artifacts/prepared_datasets

import json
manifest_path = PROJECT_DIR / 'artifacts/prepared_datasets/manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
print(json.dumps(manifest, indent=2))
assert manifest['total_examples'] == 3371, 'Unexpected count: verify the source archives.'

## Train BiomedBERT

The script creates source-grouped splits, adds capped hard examples for wrong entities, negation, and unsupported ‘first’ qualifiers, uses class-weighted loss, selects the checkpoint by validation macro-F1, and evaluates the held-out test sets once. A batch size of 4 with two accumulation steps gives an effective batch size of 8 and is safer on a T4 GPU. Training can take a while; keep the browser tab connected.

In [ ]:
assert torch.cuda.is_available(), 'GPU is not enabled. Change the Colab runtime type first.'
!biofc-train-multisource --prepared-dir artifacts/prepared_datasets --output artifacts/multisource_classifier --epochs 5 --batch-size 4 --gradient-accumulation-steps 2 --hard-negative-ratio 0.25 --fp16 --seed 42

In [ ]:
# Display only metrics produced by the completed held-out evaluation.
report_path = PROJECT_DIR / 'artifacts/multisource_classifier/training_report.json'
assert report_path.is_file(), 'Training did not finish or did not save a report.'
report = json.loads(report_path.read_text(encoding='utf-8'))
print('Split sizes:', report['split_sizes'])
print(json.dumps(report['test'], indent=2))

## Focused precision check

This five-case diagnostic tests the reported Vitamin C/Vitamin A entity confusion, negation, and the unsupported word ‘first’. It is a focused regression check, not a replacement for the untouched test datasets.

In [ ]:
!biofc-evaluate-challenge --model-dir artifacts/multisource_classifier --challenge data/challenge_sets/vitamin_c_precision.jsonl --output artifacts/evaluation/vitamin_c_precision_report.json

## Handoff

The checkpoint is saved in `artifacts/multisource_classifier/`. To use it in the Streamlit app, set `BIOFC_CLASSIFIER_DIR=artifacts/multisource_classifier` or change `classifier.model_dir` in `config/default.toml`.

Keep `training_report.json` and `test_predictions.jsonl` with the model. They record the actual split, setup, overall metrics, per-dataset metrics, and row-level predictions. Do not copy the displayed scores into the report unless this cell completed successfully.

The next evaluation should test retrieval recall and end-to-end video claims separately from verifier classification. A correct classifier cannot recover a paper that PubMed retrieval missed.